# PopOut AI: Search, Optimization, and Decision Trees

**Main delivery notebook**

This notebook presents the complete PopOut AI project at delivery level. It is designed to be readable by someone evaluating the assignment: it explains what was built, why each technical choice matters, how the algorithms work, how the experiments are reproduced, and what the main strengths and limitations are.

The project implements a playable **PopOut** game, a variant of Connect Four where players may either drop a piece into a column or pop one of their own pieces from the bottom of a column. The AI work combines:

- **Monte Carlo Tree Search with UCT** as the main adversarial search method.
- **MCTS-Solver** ideas to propagate proven wins, losses, and draws.
- **Numba optimized agents** for high-throughput search.
- A custom **ID3 decision tree**, implemented from scratch and trained from MCTS-generated PopOut data.
- CLI and GUI interfaces for human and computer play.

## Executive Summary

The project is organized around one central idea: build a correct PopOut engine first, then use it as a platform for comparing search-based and learned agents.

The **engine** uses a compact bitboard representation, which makes legal move generation and win detection efficient. On top of that engine, the project implements several **MCTS agents**, from a pure Python UCT baseline to solver-style and Numba accelerated variants. Finally, the project uses strong MCTS agents as a teacher to generate a supervised dataset for an **ID3 decision tree** agent.

This gives the project both an adversarial AI component and a machine learning component. The result is not just a game, but a small experimental framework for PopOut AI.

## Notebook Setup

The following cell makes imports work whether the notebook is opened from the repository root or directly from the `notebooks/` folder.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repository root:", ROOT)

## Assignment Coverage

| Requirement | Implementation |
|---|---|
| Playable PopOut program | `python -m src` for GUI and `python -m src --cli` for terminal play |
| Human vs Human | Supported by GUI and CLI |
| Human vs Computer | Supported by GUI and CLI |
| Computer vs Computer | Supported by CLI tournament mode |
| MCTS using UCT | `src/mcts/standard/base.py` and `src/mcts/standard/uct_standard.py` |
| Variants beyond standard MCTS | Experimental UCT, MCTS-Solver, Numba MCTS, flat Numba MCTS, optimized solver agents |
| Decision tree with ID3 | `src/decision_tree/id3/learner.py`, implemented from scratch |
| Iris decision-tree warm-up | `notebooks/ID3_Decision_Tree.ipynb` |
| PopOut dataset generated with MCTS | `src/decision_tree/dataset_generator.py` and `data/generated/popout_dt_dataset.csv` |
| Documentation and evaluation | Notebooks, tests, figures, and project summary |

A key compliance point: the submitted ID3 learner is implemented manually. External libraries may be used for data handling and evaluation, but not to train the decision tree itself.

## Repository Structure

The project is separated into clear modules:

- `src/engine/standard/`: reference bitboard engine and game rules.
- `src/engine/optimized/`: Numba optimized rule and bitboard kernels.
- `src/mcts/standard/`: pure Python MCTS, experimental UCT, and MCTS-Solver.
- `src/mcts/optimized/`: JIT accelerated search implementations.
- `src/decision_tree/`: ID3 implementation, feature generation, dataset generation, and playable ID3 agents.
- `src/interfaces/`: CLI and Pygame GUI.
- `tests/`: automated validation for the engine, agents, datasets, UI imports, and performance-sensitive pieces.
- `notebooks/`: delivery explanation and experimental analysis.
- `data/generated/`: generated datasets and trained models.
- `data/figures/`: benchmark and comparison plots.

In [ ]:
important_paths = [
    "src/engine/standard/bitboard.py",
    "src/engine/standard/rules.py",
    "src/mcts/standard/base.py",
    "src/mcts/standard/uct_solver.py",
    "src/mcts/optimized/numba_mcts.py",
    "src/decision_tree/id3/learner.py",
    "src/decision_tree/dataset_generator.py",
    "src/interfaces/cli.py",
    "src/interfaces/gui.py",
]

for path in important_paths:
    print(f"{'OK' if (ROOT / path).exists() else 'MISSING':7s} {path}")

## 1. Game Model

PopOut keeps the familiar 7 by 6 Connect Four board, but changes the action space. A player can:

- **Drop** a piece into any non-full column.
- **Pop** a bottom piece from a column, but only if that bottom piece belongs to the current player.

Moves are encoded as integers:

- `0..6`: drop in column `0..6`.
- `7..13`: pop from column `0..6`.

The pop rule makes the game more dynamic than Connect Four. A pop can create or destroy several alignments at once. If both players obtain four-in-a-row after a move, the PopOut rule gives the win to the player who made the move.

In [ ]:
from src.engine.standard.bitboard import PopOutBoard
from src.engine.standard.rules import evaluate_after_move, has_won

def describe_move(move: int) -> str:
    return f"drop_{move}" if move < 7 else f"pop_{move - 7}"

board = PopOutBoard()
for move in [3, 3, 2, 2, 4, 4, 1]:
    mover = board.current_player
    board.apply_move(move)
    winner = evaluate_after_move(board, mover=mover)

print(board)
print("Legal moves:", [describe_move(m) for m in board.legal_moves()])
print("Winner after last move:", winner)

## 2. Engine Design: Bitboards

The engine represents the board with two integer masks:

- `mask_p1`: cells occupied by player 1.
- `mask_p2`: cells occupied by player 2.

Each column uses 7 bits: 6 playable cells plus a guard bit. This representation is compact and makes win detection fast through bit shifts. The same idea is commonly used in strong Connect Four engines because it avoids scanning the board cell by cell.

Win detection checks four directions:

- vertical,
- horizontal,
- diagonal down-right,
- diagonal up-right.

This engine is the foundation of the whole project. If the rules are wrong, all AI results become unreliable; therefore the rule layer is covered by automated tests.

In [ ]:
from src.engine.standard.rules import extended_features

features = extended_features(board)
selected = [
    "current_player", "threats_me", "threats_opp",
    "center_me", "center_opp", "phase", "can_win", "opp_wins_next",
]

for key in selected:
    print(f"{key:15s}: {features[key]}")

## 3. Baseline AI: MCTS with UCT

**Monte Carlo Tree Search** is used as the main adversarial algorithm. It is a good fit for PopOut because the branching factor is manageable but the game tree is still too large for exhaustive search in normal play.

Each MCTS iteration has four phases:

1. **Selection**: descend through the tree using UCT.
2. **Expansion**: add a new child for an untried legal move.
3. **Simulation**: play a rollout until win, draw, repetition, or depth limit.
4. **Backpropagation**: update visit counts and value estimates.

The UCT score balances exploitation and exploration:

`score = Q + C * sqrt(log(parent_visits) / child_visits)`

where `Q` is the mean reward and `C` controls exploration. The baseline agent chooses the final move by visit count, which is a standard robust choice for MCTS.

In [ ]:
from src.mcts.factory import get_agent

agent = get_agent("standard", seed=7, rollout_depth=100)
move = agent.run(PopOutBoard(), iterations=300)

print("Standard UCT move from the empty board:", describe_move(move))

## 4. MCTS Variants

The project does not stop at a single baseline. It includes several agents behind the same factory interface:

| Agent name | Purpose |
|---|---|
| `standard` | Pure Python UCT baseline |
| `experimental` | Small selection variation for comparison |
| `solver` | MCTS-Solver with proof propagation |
| `numba` | JIT optimized MCTS |
| `flat_numba` | Faster flat-array Numba MCTS |
| `numba_solver` | Optimized solver-style MCTS |
| `flat_numba_solver` | Fastest solver-style variant |
| `id3` | Playable ID3 agent with tactical safeguards |
| `id3_raw` | ID3 agent with raw board-style features |

This design keeps the interfaces simple: experiments and games can switch agents by name.

In [ ]:
agent_names = [
    "standard", "experimental", "solver",
    "numba", "flat_numba", "numba_solver", "flat_numba_solver",
    "id3", "id3_raw",
]

for name in agent_names:
    try:
        candidate = get_agent(name, seed=1)
        print(f"{name:18s} -> {candidate.__class__.__name__}")
    except Exception as exc:
        print(f"{name:18s} -> not available in this environment ({type(exc).__name__}: {exc})")

## 5. MCTS-Solver

Standard MCTS estimates move quality statistically. **MCTS-Solver** adds game-theoretic proof information to the tree.

Each node can be marked as:

- `UNKNOWN`: not yet proved.
- `WIN`: the player to move has a forced win.
- `LOSS`: the player to move is losing with perfect play.
- `DRAW`: the position is a forced draw.

The solver propagates these labels upward using AND/OR game-tree logic:

- A node is a **WIN** if at least one child is a **LOSS** for the opponent.
- A node is a **LOSS** if all legal children are **WIN** for the opponent.
- A node is a **DRAW** if no win is possible but a draw can be forced.

It also stores a minimax distance, so it prefers faster wins and slower losses. This is an important improvement over a purely statistical agent because some moves become mathematically justified rather than only empirically promising.

## 6. Performance Engineering

The optimized implementation moves hot loops into Numba-compiled functions. This matters because MCTS strength is closely tied to the number of iterations it can run within a time budget.

The documentation reports approximate throughput improvements such as:

| Engine | Approximate throughput |
|---|---:|
| `StandardUCT` | about 9k iterations/s |
| `SolverMCTS` | about 8k iterations/s |
| `NumbaMCTS` | about 50k iterations/s |
| `NumbaSolverMCTS` | about 37k iterations/s |
| `FlatNumbaMCTS` | about 179k iterations/s |
| `FlatNumbaSolverMCTS` | about 168k iterations/s |

The exact numbers depend on hardware and environment, but the conclusion is stable: the optimized agents allow much deeper search in practical play.

In [ ]:
figures_dir = ROOT / "data" / "figures"
if figures_dir.exists():
    for fig in sorted(figures_dir.glob("*.png")):
        print(fig.relative_to(ROOT))
else:
    print("No figures directory found.")

In [ ]:
from IPython.display import Image, display

for name in ["all_engines_throughput.png", "mcts_iterations.png", "mcts_exploration_c.png"]:
    path = figures_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 7. Decision Tree: ID3 From Scratch

The decision-tree component implements ID3 manually in `src/decision_tree/id3/learner.py`.

The core learning process is:

1. Compute the entropy of the target labels.
2. For each candidate feature, compute information gain.
3. Split on the feature with highest information gain.
4. Recurse until the node is pure, no features remain, or the maximum depth is reached.
5. Store majority labels for robust prediction when an unseen feature value appears.

This satisfies the assignment requirement because the tree induction algorithm is implemented directly rather than delegated to a library classifier.

In [ ]:
import pandas as pd
from src.decision_tree.id3.learner import ID3Classifier

toy = pd.DataFrame({
    "center_control": ["low", "high", "high", "low", "high", "low"],
    "can_win": ["no", "yes", "no", "no", "yes", "yes"],
    "move": ["drop_3", "drop_4", "drop_3", "drop_2", "drop_4", "pop_0"],
})

tree = ID3Classifier(max_depth=2)
tree.fit(toy, target="move")
print(tree.tree_to_string())

## 8. PopOut Dataset Generation

The PopOut decision tree is trained from states labelled by MCTS. Each row describes a board position and the target is the move selected by the search agent.

The dataset includes:

- 42 board cell features.
- the current player,
- tactical features such as threats, center control, phase, immediate win, and opponent immediate win,
- the target move `best_move`,
- optional metadata such as whether the oracle position was proven.

Horizontal mirroring is used as data augmentation because PopOut is symmetric left-to-right.

In [ ]:
dataset_path = ROOT / "data" / "generated" / "popout_dt_dataset.csv"

if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    print("Dataset:", dataset_path.relative_to(ROOT))
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Target column:", "best_move" in df.columns)
    print("Proven metadata:", "is_proven" in df.columns)
    display(df.head(3))
else:
    print("Dataset not found. It can be generated with src/decision_tree/dataset_generator.py.")

In [ ]:
if dataset_path.exists() and "best_move" in df.columns:
    label_counts = df["best_move"].value_counts().sort_index()
    print("Number of labels:", len(label_counts))
    display(label_counts)

    drop_count = df["best_move"].str.startswith("drop_").sum()
    pop_count = df["best_move"].str.startswith("pop_").sum()
    print(f"Drop labels: {drop_count}")
    print(f"Pop labels:  {pop_count}")

    if "is_proven" in df.columns:
        print("Proven positions:", int(df["is_proven"].sum()))

## 9. Playable ID3 Agent

The project includes two playable ID3 agents:

- `ID3Agent`: uses board cells plus tactical features.
- `ID3AgentRaw`: uses a simpler raw feature representation.

The playable `ID3Agent` is intentionally safer than a pure classifier. Before asking the tree for a prediction, it checks for immediate winning moves and immediate blocks. This should be reported honestly: the classifier is learned with ID3, while the playable game agent is a hybrid of ID3 prediction and small tactical rules.

That hybrid design is reasonable for gameplay because a tree trained to imitate MCTS can still miss rare tactical emergencies.

## 10. Evaluation Strategy

The project uses several kinds of evaluation:

- **Correctness tests**: validate game rules, legal moves, win detection, repetition, and board state behavior.
- **Agent tests**: check MCTS, solver behavior, optimized search, and ID3 prediction paths.
- **Performance tests and figures**: compare throughput across engines.
- **Decision-tree experiments**: evaluate ID3 on Iris and PopOut datasets.
- **Gameplay experiments**: compare agents in computer-vs-computer scenarios.

For a stochastic adversarial algorithm like MCTS, one game is not enough. Results should be interpreted through repeated matches, alternating first player, fixed seeds when needed, and clear iteration budgets.

In [ ]:
test_files = sorted((ROOT / "tests").glob("test_*.py"))
print("Number of test files:", len(test_files))
for path in test_files:
    print("-", path.relative_to(ROOT))

## 11. Reproducibility

The recommended environment is Conda:

```bash
conda env create -f environment.yml
conda activate popout-ai
pytest -q
```

Run the graphical interface:

```bash
python -m src
```

Run the command-line interface:

```bash
python -m src --cli
```

CLI move notation:

- `d3`: drop in column 3.
- `p0`: pop from column 0.

A small dataset can be regenerated with the dataset generator. Large oracle-quality datasets are more expensive because they rely on strong MCTS searches.

In [ ]:
env_path = ROOT / "environment.yml"
if env_path.exists():
    print(env_path.read_text())

## 12. Limitations and Honest Notes

A strong delivery should also state limitations clearly:

- **Environment dependence**: optimized agents require Numba and the intended Python environment.
- **Stochasticity**: MCTS results depend on seed, iteration budget, rollout depth, and exploration constant.
- **First-player effects**: head-to-head evaluations should alternate colors.
- **Hybrid ID3 agent**: the playable ID3 agent includes tactical safeguards, so it is not a purely tree-only policy.
- **Dataset bias**: a tree trained from MCTS data can inherit the strengths and weaknesses of the MCTS oracle.
- **PopOut draw handling**: repetition and full-board behavior are implemented, but should be explained as part of the AI design.

These points do not weaken the project; they make the methodology clearer and more credible.

## 13. Why This Is a Strong AI Project

The project is strong because the components support each other:

- The bitboard engine gives fast and reliable game operations.
- MCTS provides a general adversarial strategy without handcrafted evaluation functions.
- MCTS-Solver adds exact proof propagation where the tree is sufficiently explored.
- Numba optimization turns algorithmic work into practical playing strength.
- The ID3 pipeline converts search behavior into a supervised model.
- The interfaces make the system usable beyond notebooks.
- The test suite increases confidence that the engine and agents behave consistently.

In other words, the project has both breadth and depth: a playable application, multiple AI approaches, performance work, machine learning, and validation.

## Final Submission Checklist

Before presenting or submitting, check:

- The Conda environment is active.
- `pytest -q` runs in the intended environment.
- The main notebooks have no saved error outputs.
- The README commands match the actual module paths.
- The distinction between pure ID3 training and hybrid playable ID3 behavior is explicit.
- Any use of external libraries is clearly separated from the custom ID3 learner.
- Performance figures are generated or included.
- Computer-vs-computer claims use repeated games and alternate first player when possible.

## Conclusion

This project implements PopOut as a complete AI system. It starts with a correct and efficient rule engine, builds several MCTS-based agents on top of it, improves search with solver logic and Numba optimization, and then uses MCTS as a teacher for a custom ID3 decision-tree agent.

The final result satisfies the assignment requirements while also going beyond the minimum: it includes multiple agent variants, a generated supervised dataset, a playable GUI and CLI, optimized search, and a broad test suite.